# Check Array Job Outputs

This notebook scans an array-job output directory with structure:

- `task_config_map.yaml`
- `task_1`, `task_2`, ...

It classifies each task as:

- **completed**: contains both `details.yaml` and `idata.nc`
- **partial**: contains `idata.nc` but missing `details.yaml`
- **empty**: task folder exists but has no files
- **failed**: task folder exists but does not meet any of the above

In [ ]:
from pathlib import Path
import re
import pandas as pd

In [ ]:
# Set this to the array-job output directory you want to check.
# Example:
# base_dir = Path('../remote_cluster/outputs/58681106_array_job')
base_dir = Path('../remote_cluster/outputs/58744145_array_job_2')

if not base_dir.exists():
    raise FileNotFoundError(f'Directory not found: {base_dir.resolve()}')

print('Scanning:', base_dir.resolve())

In [ ]:
task_dirs = [
    p for p in base_dir.iterdir()
    if p.is_dir() and re.fullmatch(r'task_\d+', p.name)
]
task_dirs = sorted(task_dirs, key=lambda p: int(p.name.split('_')[1]))

if not task_dirs:
    raise ValueError(f'No task folders found in {base_dir.resolve()}')

records = []
for task_dir in task_dirs:
    files = [x for x in task_dir.iterdir() if x.is_file()]
    file_names = {f.name for f in files}

    has_idata = 'idata.nc' in file_names
    has_details = 'details.yaml' in file_names

    if len(files) == 0:
        status = 'empty'
    elif has_idata and has_details:
        status = 'completed'
    elif has_idata:
        status = 'partial'
    else:
        status = 'failed'

    records.append({
        'task': task_dir.name,
        'n_files': len(files),
        'has_idata_nc': has_idata,
        'has_details_yaml': has_details,
        'status': status,
    })

results = pd.DataFrame(records)
results

In [ ]:
summary = (
    results['status']
    .value_counts()
    .rename_axis('status')
    .reset_index(name='count')
)

summary

In [ ]:
completed_tasks = results.loc[results['status'] == 'completed', 'task'].tolist()
partial_tasks = results.loc[results['status'] == 'partial', 'task'].tolist()
empty_tasks = results.loc[results['status'] == 'empty', 'task'].tolist()
failed_tasks = results.loc[results['status'] == 'failed', 'task'].tolist()

print(f'Total tasks scanned: {len(results)}')
print(f'Completed: {len(completed_tasks)}')
print(f'Partial (idata.nc only): {len(partial_tasks)}')
print(f'Empty: {len(empty_tasks)}')
print(f'Failed (non-empty but missing idata.nc): {len(failed_tasks)}')

In [ ]:
status_lists = {
    'completed': completed_tasks,
    'partial': partial_tasks,
    'empty': empty_tasks,
    'failed': failed_tasks,
}

for label, tasks in status_lists.items():
    print(f'\n{label.upper()} ({len(tasks)}):')
    if tasks:
        print(', '.join(tasks))
    else:
        print('None')

In [ ]:
# Optional: save per-task status to CSV
output_csv = base_dir / 'task_status_summary.csv'
results.to_csv(output_csv, index=False)
print('Saved:', output_csv)

In [ ]:
import yaml

config_map_path = base_dir / 'task_config_map.yaml'
if not config_map_path.exists():
    raise FileNotFoundError(f'Config map not found: {config_map_path}')

with open(config_map_path, 'r') as f:
    raw_task_config_map = yaml.safe_load(f)

config_source = raw_task_config_map.get('tasks', raw_task_config_map) if isinstance(raw_task_config_map, dict) else raw_task_config_map

def get_task_config(task_name):
    task_num = int(task_name.split('_')[1])
    return config_source['task_to_config'][task_num]

task_configs_by_status = {
    status_name: [
        {'task': task_name, 'config': get_task_config(task_name)}
        for task_name in task_list
    ]
    for status_name, task_list in status_lists.items()
}

for status_name, items in task_configs_by_status.items():
    print(f'\n{status_name.upper()} ({len(items)}):')
    if not items:
        print('None')
        continue

    for item in items:
        print(f"\n{item['task']}:")
        if item['config'] is None:
            print('  <config not found>')
            continue

        config_text = yaml.safe_dump(item['config'], sort_keys=False).rstrip()
        print('\n'.join(f'  {line}' for line in config_text.splitlines()))